# 02 - Cổng nhiều qubit và entanglement

Một qubit thì chỉ có superposition. Sức mạnh thật sự xuất hiện khi có **nhiều qubit**:

- $n$ qubit cần $2^n$ amplitude để mô tả - không gian lớn theo hàm mũ.
- Có những trạng thái **không tách được** thành tích của từng qubit riêng lẻ - đó là **entanglement**.

Bài này đi từ tích tensor -> CNOT -> trạng thái Bell -> đo entanglement -> các cổng controlled khác.

In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit
import qiskit_aer

print("qiskit:", qiskit.__version__, "| qiskit-aer:", qiskit_aer.__version__)


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def probs(sv, nd=3):
    """Xác suất đo được từng kết quả, key là chuỗi bit thường (không phải np.str_)."""
    return {str(k): round(float(v), nd) for k, v in sv.probabilities_dict().items()}


from qiskit.quantum_info import DensityMatrix, SparsePauliOp, entropy, partial_trace
from qiskit_aer.primitives import EstimatorV2

## 1. Tích tensor và thứ tự qubit

Hai qubit độc lập ghép lại bằng **tích tensor** (Kronecker):

$$|a\rangle \otimes |b\rangle, \qquad
  \begin{pmatrix}a_0\\a_1\end{pmatrix} \otimes \begin{pmatrix}b_0\\b_1\end{pmatrix}
  = \begin{pmatrix}a_0b_0\\a_0b_1\\a_1b_0\\a_1b_1\end{pmatrix}$$

Quy ước Qiskit (little-endian): trạng thái viết là $|q_{n-1} \dots q_1 q_0\rangle$, tức
**qubit 0 đứng bên phải**. Nên `Statevector.from_label("10")` nghĩa là qubit 1 = 1, qubit 0 = 0,
và tích tensor phải viết là `q1.tensor(q0)`.

In [ ]:
q0 = Statevector.from_label("0")  # qubit 0
q1 = Statevector.from_label("1")  # qubit 1

tich = q1.tensor(q0)  # |q1> (x) |q0>
print("q1.tensor(q0)         :", tich.data.real, "->", ket(tich))
print("np.kron(q1, q0)       :", np.kron(q1.data, q0.data).real)
print("Statevector 'from 10' :", Statevector.from_label("10").data.real)
print("\nThứ tự chỉ số của 2 qubit: 00, 01, 10, 11  (trái = qubit 1, phải = qubit 0)")

## 2. CNOT (CX) - cổng 2 qubit quan trọng nhất

CNOT có một qubit **control** và một qubit **target**:
nếu control = 1 thì lật target, ngược lại không làm gì.

| vào $\lvert q_1q_0\rangle$ | ra |
|---|---|
| $\lvert 00\rangle$ | $\lvert 00\rangle$ |
| $\lvert 01\rangle$ | $\lvert 11\rangle$ |
| $\lvert 10\rangle$ | $\lvert 10\rangle$ |
| $\lvert 11\rangle$ | $\lvert 01\rangle$ |

(bảng trên ứng với `qc.cx(0, 1)`: control = qubit 0 = ký tự phải.)

CNOT là cổng tạo **tương quan** giữa hai qubit - và đó cũng là cổng phần cứng khó làm nhất,
nên số lượng CX là thước đo chi phí chính khi chạy trên máy thật.

In [ ]:
cx = QuantumCircuit(2)
cx.cx(0, 1)  # control = qubit 0, target = qubit 1
print(cx.draw())

print("\nma trận CX (cơ sở 00, 01, 10, 11):")
print(Operator(cx).data.real.astype(int))

print()
for nhan in ["00", "01", "10", "11"]:
    s = Statevector.from_label(nhan)
    print(f"CX|{nhan}> = {ket(s.evolve(cx))}")

## 3. Trạng thái Bell - ví dụ chuẩn của entanglement

Chỉ cần hai cổng:

$$|00\rangle \xrightarrow{H \text{ trên } q_0} \tfrac{1}{\sqrt2}(|00\rangle + |01\rangle)
  \xrightarrow{CX_{0\to1}} \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle) = |\Phi^+\rangle$$

Kết quả: chỉ có `00` và `11`, **không bao giờ** có `01` hay `10`. Hai qubit luôn cho cùng kết quả,
dù đo ở đâu - và điều này **không** giải thích được bằng "mỗi qubit có xác suất riêng".

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
print(bell.draw())

sv_bell = Statevector(bell)
print("\ntrạng thái:", ket(sv_bell))
print("xác suất  :", probs(sv_bell))

do_bell = bell.copy()
do_bell.measure_all()
counts = SamplerV2().run([do_bell], shots=1000).result()[0].data.meas.get_counts()
print("counts    :", counts)
display(plot_histogram(counts))

### Bốn trạng thái Bell

$$|\Phi^\pm\rangle = \tfrac{1}{\sqrt2}(|00\rangle \pm |11\rangle), \qquad
  |\Psi^\pm\rangle = \tfrac{1}{\sqrt2}(|01\rangle \pm |10\rangle)$$

Bốn trạng thái này trực giao nhau và tạo thành một cơ sở của không gian 2 qubit
(**Bell basis**). Cách tạo: thêm $X$ vào qubit 0 và/hoặc qubit 1 **trước** mạch $H$ + $CX$.

Kết quả ở dưới có thể lệch một dấu tổng thể so với công thức trên (ví dụ $\Psi^-$ ra
$\frac{1}{\sqrt2}(|10\rangle - |01\rangle)$) - đó là **global phase**, không quan sát được,
nên vẫn đúng là trạng thái đó.

In [ ]:
chuan_bi = {
    "Phi+": [],
    "Phi-": [0],  # X trên qubit 0 trước
    "Psi+": [1],  # X trên qubit 1 trước
    "Psi-": [0, 1],
}

for ten, truoc in chuan_bi.items():
    qc = QuantumCircuit(2)
    for q in truoc:
        qc.x(q)
    qc.h(0)
    qc.cx(0, 1)
    print(f"{ten}: {ket(Statevector(qc))}")

## 4. Làm sao biết một trạng thái có entangled hay không?

Trạng thái **separable** (tích) viết được thành $|\psi_A\rangle \otimes |\psi_B\rangle$;
trạng thái **entangled** thì không.

Cách kiểm tra định lượng: lấy **partial trace** để bỏ một qubit đi, được
density matrix $\rho_0$ của qubit còn lại, rồi xét:

- **Purity** $\mathrm{Tr}(\rho_0^2)$: bằng 1 nếu qubit ở pure state, $0.5$ nếu entangled tối đa (với 1 qubit).
- **Entropy von Neumann** $S(\rho_0)$: bằng 0 nếu không entangled, bằng 1 nếu entangled tối đa.

Ý nghĩa: nếu hai qubit entangled với nhau thì **riêng từng qubit không có trạng thái xác định** -
toàn bộ thông tin nằm ở mối tương quan giữa chúng.

In [ ]:
tich_qc = QuantumCircuit(2)
tich_qc.h(0)
tich_qc.h(1)  # H (x) H -> trạng thái separable

for ten, qc in [("Separable (H trên cả hai)", tich_qc), ("Entangled (Bell Phi+)", bell)]:
    sv = Statevector(qc)
    rho0 = partial_trace(sv, [1])  # bỏ qubit 1, giữ qubit 0
    print(ten)
    print("   trạng thái :", ket(sv))
    print("   rho(qubit 0) =\n", np.round(rho0.data, 3))
    print("   purity  :", round(float(rho0.purity().real), 3))
    print("   entropy :", round(float(entropy(rho0)), 3), "\n")

### Tương quan nhìn qua expectation value

`EstimatorV2` cho giá trị kỳ vọng $\langle\psi|O|\psi\rangle$ của một observable,
không cần thêm lệnh đo. Với $|\Phi^+\rangle$:

- $\langle ZZ\rangle = +1$ - hai qubit luôn cùng kết quả trong cơ sở Z.
- $\langle XX\rangle = +1$ - vẫn tương quan hoàn hảo **cả trong cơ sở X** (đây mới là điểm lạ;
  tương quan cổ điển thông thường không làm được điều này).
- $\langle ZI\rangle = 0$ - nhưng riêng từng qubit thì hoàn toàn ngẫu nhiên.

In [ ]:
est = EstimatorV2()
for ten_obs in ["ZZ", "XX", "YY", "ZI", "IX"]:
    ev = est.run([(bell, SparsePauliOp(ten_obs))]).result()[0].data.evs
    print(f"<{ten_obs}> = {float(ev):+.3f}")

## 5. Các cổng nhiều qubit khác

| Cổng | Qiskit | Ý nghĩa |
|---|---|---|
| CX / CNOT | `qc.cx(c, t)` | control = 1 thì lật target |
| CZ | `qc.cz(a, b)` | cả hai = 1 thì thêm dấu âm; **đối xứng** |
| SWAP | `qc.swap(a, b)` | đổi chỗ hai qubit |
| Toffoli (CCX) | `qc.ccx(c1, c2, t)` | hai control; đủ để làm mọi cổng logic cổ điển |
| CH, CRX, CP... | `qc.ch(c, t)`, `qc.crx(th, c, t)` | bản controlled của cổng 1 qubit |

Quan hệ đáng nhớ: kẹp $H$ lên **target** ngay trước và ngay sau $CX$ thì được $CZ$ - cổng
controlled đổi "kiểu" khi ta đổi cơ sở của target. Viết bằng ma trận thì phải theo quy ước
little-endian của Qiskit: ở đây target là qubit 1 nên $H$ nằm ở **thừa số bên trái**,
$CZ = (H \otimes I)\, CX\, (H \otimes I)$.

In [ ]:
cz = QuantumCircuit(2)
cz.cz(0, 1)
print("CZ =\n", Operator(cz).data.real.astype(int))

tuong_duong = QuantumCircuit(2)
tuong_duong.h(1)
tuong_duong.cx(0, 1)
tuong_duong.h(1)
print("\nCZ == H-CX-H trên target ?", Operator(cz) == Operator(tuong_duong))

sw = QuantumCircuit(2)
sw.swap(0, 1)
print("\nSWAP|01> =", ket(Statevector.from_label("01").evolve(sw)))

tof = QuantumCircuit(3)
tof.ccx(0, 1, 2)  # control = qubit 0 và 1, target = qubit 2
print("\nBảng chân trị Toffoli (|q2 q1 q0>):")
for nhan in ["000", "001", "010", "011", "100", "101", "110", "111"]:
    print(f"  CCX|{nhan}> = {ket(Statevector.from_label(nhan).evolve(tof))}")

### Tự tạo cổng controlled từ một cổng bất kỳ

`.to_gate().control(n)` biến một mạch con thành cổng có $n$ đường control.

In [ ]:
con = QuantumCircuit(1)
con.ry(np.pi / 2, 0)
cong_dieu_khien = con.to_gate(label="RY(pi/2)").control(1)

qc = QuantumCircuit(2)
qc.x(0)  # bật control lên 1 để thấy tác dụng
qc.append(cong_dieu_khien, [0, 1])
print(qc.draw())
print("\nkết quả:", ket(Statevector(qc)))

## 6. GHZ - entangled nhiều hơn 2 qubit

$$|GHZ_n\rangle = \tfrac{1}{\sqrt2}(|0\dots0\rangle + |1\dots1\rangle)$$

Mạch: một $H$ rồi một chuỗi $CX$ nối tiếp. Đo $n$ qubit chỉ ra được toàn 0 hoặc toàn 1.

In [ ]:
def ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    return qc


g = ghz(4)
print(g.draw())
print("\ntrạng thái:", ket(Statevector(g)))

gm = g.copy()
gm.measure_all()
counts = SamplerV2().run([gm], shots=1000).result()[0].data.meas.get_counts()
print("counts:", counts)
display(plot_histogram(counts))

## Tóm tắt

- Tích tensor ghép các qubit: $n$ qubit -> $2^n$ amplitude; Qiskit xếp bit kiểu little-endian.
- `cx` là cổng tạo tương quan; $H$ rồi $CX$ cho trạng thái Bell.
- Trạng thái **entangled** không tách được thành tích; kiểm tra bằng `partial_trace` + `entropy`
  (entropy > 0 nghĩa là có entanglement).
- `EstimatorV2` đo tương quan qua expectation value ($\langle ZZ\rangle$, $\langle XX\rangle$).
- Cổng controlled bất kỳ: `.to_gate().control(n)`.

### Thử tự làm

1. Đổi `bell.cx(0, 1)` thành `bell.cz(0, 1)` - còn entangled không? Dùng entropy để kiểm tra.
2. Tạo $|\Psi^-\rangle$ và tính $\langle ZZ\rangle$, $\langle XX\rangle$ - giải thích dấu.
3. Viết hàm `ghz(n)` phiên bản dùng cây nhị phân (độ sâu $\log n$ thay vì $n$) và so sánh `depth()`.

Bài tiếp theo: `03_qiskit_workflow.ipynb`.